# CV Eclipse Timing & O-C Analysis Pipeline v5.0

## Overview

This notebook provides a comprehensive pipeline for analyzing eclipsing cataclysmic variables (CVs) using TESS data. The pipeline includes:

1. **Data Loading**: TESS lightcurve data acquisition
2. **Outburst Detection**: Automated identification and removal of outbursts
3. **Period Search**: Interactive periodogram analysis with BLS and Lomb-Scargle methods
4. **Eclipse Timing**: Precise measurement of eclipse mid-times
5. **O-C Analysis**: Orbital period evolution and timing residuals

### Key Features:
- **Interactive Period Selection**: Click-to-select periodogram interface
- **Automated Outburst Masking**: Sigma-clipping based detection
- **Template Cross-Correlation**: High-precision eclipse timing
- **Comprehensive Diagnostics**: Visual analysis tools and statistics

In [ ]:
# %matplotlib widget

# Configuration
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
from modules import build_template, measure_midpoints
import warnings
warnings.filterwarnings('ignore')
from lightkurve import LightCurve

# Target Configuration
TIC_ID = 219107776
TARGET_NAME = "EX Dra"
OUTPUT_DIR = "output"
SECTOR = None

# Analysis Parameters
PERIOD_MIN = 0.05
PERIOD_MAX = 0.5
MIN_ECLIPSE_DEPTH = 0.05
ECLIPSE_WIDTH = 0.1
MIN_CYCLE_COVERAGE = 0.5
TIMING_OUTLIER_SIGMA = 3.0
NUM_OF_SECTORS = 10

print(f"Target: {TARGET_NAME} (TIC {TIC_ID})")
print(f"Period search: {PERIOD_MIN} - {PERIOD_MAX} days")
print(f"Min eclipse depth: {MIN_ECLIPSE_DEPTH*100:.0f}%")

## Step 1: Configuration & Setup

Configure the analysis parameters for your target. Modify the values below according to your specific CV target.

In [ ]:
# Data Loading
print("Loading TESS data...")
search_result = lk.search_lightcurve(f'TIC {TIC_ID}', mission='TESS', sector=SECTOR)
if len(search_result) == 0:
    raise ValueError(f"No TESS data found for TIC {TIC_ID}")

lc_collection = search_result.download_all()

if type(NUM_OF_SECTORS) == int:
    lc = lc_collection[:NUM_OF_SECTORS].stitch()
else:
    lc = lc_collection.stitch()
lc : LightCurve = lc.remove_nans().remove_outliers(sigma=5)

print(f"✓ Loaded {len(lc)} data points from {len(lc_collection)} sectors")
print(f"  Time span: {lc.time.max() - lc.time.min()} days")

In [ ]:
lc_clean, mask = lc.remove_outliers(sigma=3, return_mask=True)
lc_not_clean = lc[mask]

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(lc_clean.time.value, lc_clean.flux.value, s=0.5, alpha=0.7, color='blue', label='Clean data')
ax.scatter(lc_not_clean.time.value, lc_not_clean.flux.value, s=0.5, alpha=0.7, color='red', label='Outliers removed')
ax.set_xlabel('Time (BJD_TDB)')
ax.set_ylabel('Flux')
ax.set_title(f'{TARGET_NAME} - Outlier Removal Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Data Preprocessing
print("Preprocessing data...")

# Remove outliers and normalize
lc_clean = lc.remove_outliers(sigma=3)
lc_clean = lc_clean.normalize()

# Remove outbursts (simple moving median filter)
window = int(0.5 / np.median(np.diff(lc_clean.time.value)))  # 0.5 day window
rolling_median = np.convolve(lc_clean.flux.value, np.ones(window)/window, mode='same')
outburst_mask = np.abs(lc_clean.flux.value - rolling_median) > 0.1
lc_clean = lc_clean[~outburst_mask]

print(f"✓ Cleaned data: {len(lc_clean)} points ({len(lc) - len(lc_clean)} removed)")

# Quick visualization
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lc_clean.time.value, lc_clean.flux.value, '.', markersize=0.5, alpha=0.7)
ax.set_xlabel('Time (BJD_TDB)')
ax.set_ylabel('Normalized Flux')
ax.set_title(f'{TARGET_NAME} - Cleaned Light Curve')
plt.tight_layout()
plt.show()

---

## Step 2: Data Loading

Load TESS lightcurve data for your target. You can either:
1. **Load TESS data directly** using the TIC ID (recommended)
2. **Load from file** if you have pre-processed data

The pipeline will automatically handle data validation and provide summary statistics.

In [ ]:
# Period Search
print("Searching for orbital period...")

# Calculate periodogram
periodogram = lc_clean.to_periodogram(method='lombscargle', 
                                      minimum_period=PERIOD_MIN, 
                                      maximum_period=PERIOD_MAX)

# Find best period
best_period = periodogram.period_at_max_power.value
max_power = periodogram.max_power.value

print(f"✓ Best period: {best_period:.6f} days")
print(f"  Power: {max_power:.3f}")

# Visualize periodogram
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(periodogram.period.value, periodogram.power.value, 'b-', linewidth=0.8)
ax.axvline(best_period, color='red', linestyle='--', label=f'Best: {best_period:.6f} d')
ax.set_xlabel('Period (days)')
ax.set_ylabel('Power')
ax.set_title('Periodogram')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

SELECTED_PERIOD = best_period

In [ ]:
# Eclipse Detection
print("Detecting eclipse phase...")

# Phase-fold the data
folded = lc_clean.fold(period=SELECTED_PERIOD)
phase_folded = folded.phase.value
flux_folded = folded.flux.value

# Sort by phase
sort_idx = np.argsort(phase_folded)
phase_folded = phase_folded[sort_idx]
flux_folded = flux_folded[sort_idx]

# Bin the data to find eclipse
n_bins = 100
phase_bins = np.linspace(-0.5, 0.5, n_bins)
binned_flux = np.zeros(n_bins-1)

for i in range(n_bins-1):
    mask = (phase_folded >= phase_bins[i]) & (phase_folded < phase_bins[i+1])
    if np.sum(mask) > 0:
        binned_flux[i] = np.median(flux_folded[mask])
    else:
        binned_flux[i] = 1.0

# Find eclipse center (minimum flux)
eclipse_idx = np.argmin(binned_flux)
eclipse_phase = (phase_bins[eclipse_idx] + phase_bins[eclipse_idx+1]) / 2
eclipse_depth = 1 - binned_flux[eclipse_idx]

# Adjust phase so eclipse is at 0
if eclipse_phase != 0:
    phase_folded = phase_folded - eclipse_phase
    phase_folded = np.where(phase_folded < -0.5, phase_folded + 1, phase_folded)
    phase_folded = np.where(phase_folded > 0.5, phase_folded - 1, phase_folded)

ECLIPSE_PHASE_CENTER = 0.0
print(f"✓ Eclipse depth: {eclipse_depth:.3f}")
print(f"  Eclipse phase: {eclipse_phase:.3f} (adjusted to 0.0)")

In [ ]:
# Visualize Phase-folded Light Curve
fig, ax = plt.subplots(figsize=(10, 5))

# Plot phase-folded data
ax.plot(phase_folded, flux_folded, '.', markersize=1, alpha=0.5, color='blue')

# Plot binned data
bin_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
ax.plot(bin_centers, binned_flux, 'ro-', markersize=4, linewidth=2, label='Binned')

# Mark eclipse
ax.axvline(0, color='red', linestyle='--', alpha=0.7, label='Eclipse Center')
ax.axhspan(1-eclipse_depth, 1, alpha=0.2, color='red', label=f'Eclipse Depth: {eclipse_depth:.3f}')

ax.set_xlabel('Phase')
ax.set_ylabel('Normalized Flux')
ax.set_title(f'{TARGET_NAME} - Phase-folded Light Curve (P={SELECTED_PERIOD:.6f} d)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 0.5)
plt.tight_layout()
plt.show()

---

## Step 3: Outburst Detection & Removal

Cataclysmic variables often show outbursts that can interfere with eclipse timing measurements. We'll use sigma-clipping to identify and remove these events, keeping only the quiescent (quiet) data for period analysis.

**Process:**
1. Apply sigma-clipping to identify flux outliers
2. Expand the mask to remove nearby points
3. Optionally apply CBV correction for systematics
4. Visualize the cleaned data

In [ ]:
# Prepare Eclipse Cycles
print("Preparing eclipse cycles...")

time = lc_clean.time.value
flux = lc_clean.flux.value

# Calculate cycle parameters
t0 = time[0]
time_span = time[-1] - time[0]
n_cycles = int(time_span / SELECTED_PERIOD)

# Extract individual cycles
eclipse_cycles = []
for i in range(n_cycles):
    cycle_start = t0 + i * SELECTED_PERIOD
    cycle_end = cycle_start + SELECTED_PERIOD
    
    # Get data for this cycle
    cycle_mask = (time >= cycle_start) & (time < cycle_end)
    if np.sum(cycle_mask) < 10:  # Need minimum points
        continue
    
    cycle_time = time[cycle_mask]
    cycle_flux = flux[cycle_mask]
    cycle_phase = (cycle_time - cycle_start) / SELECTED_PERIOD
    
    # Check if eclipse is present
    eclipse_mask = np.abs(cycle_phase - ECLIPSE_PHASE_CENTER) <= ECLIPSE_WIDTH/2
    if np.sum(eclipse_mask) < 5:
        continue
    
    eclipse_flux = cycle_flux[eclipse_mask]
    eclipse_depth = 1 - np.min(eclipse_flux)
    
    if eclipse_depth > MIN_ECLIPSE_DEPTH:
        eclipse_cycles.append({
            'phase': cycle_phase,
            'flux': cycle_flux,
            'cycle_start': cycle_start,
            'eclipse_depth': eclipse_depth,
            'has_eclipse': True
        })

print(f"✓ Found {len(eclipse_cycles)} cycles with eclipses out of {n_cycles} total")

In [ ]:
# Quality Assessment
print("Assessing cycle quality...")

cycle_info = []
for i, cycle in enumerate(eclipse_cycles):
    phase = cycle['phase']
    flux = cycle['flux']
    
    # Basic quality metrics
    phase_coverage = len(phase) / (1.0 / np.median(np.diff(time)))  # Fraction of expected points
    flux_scatter = np.std(flux)
    eclipse_depth = cycle['eclipse_depth']
    
    # Quality flag
    quality_good = (phase_coverage >= MIN_CYCLE_COVERAGE and 
                   eclipse_depth >= MIN_ECLIPSE_DEPTH and
                   flux_scatter < 0.1)
    
    cycle_info.append({
        'has_eclipse': cycle['has_eclipse'],
        'eclipse_depth': eclipse_depth,
        'phase_coverage': phase_coverage,
        'flux_scatter': flux_scatter,
        'quality_good': quality_good
    })

cycle_info = np.array(cycle_info)
good_cycles = sum([c['quality_good'] for c in cycle_info])
print(f"✓ {good_cycles}/{len(eclipse_cycles)} cycles meet quality criteria")

In [ ]:
# Visualize Eclipse Examples
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Show first 6 good cycles
good_indices = [i for i, c in enumerate(cycle_info) if c['quality_good']][:6]

for i, cycle_idx in enumerate(good_indices):
    cycle = eclipse_cycles[cycle_idx]
    info = cycle_info[cycle_idx]
    
    ax = axes[i]
    ax.plot(cycle['phase'], cycle['flux'], 'o-', markersize=3, alpha=0.7)
    ax.axvline(ECLIPSE_PHASE_CENTER, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'Cycle {cycle_idx+1}\nDepth: {info["eclipse_depth"]:.3f}')
    ax.set_xlabel('Phase')
    ax.set_ylabel('Flux')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Step 4: Period Search & Selection 🔍
---

This step performs an interactive period search using multiple methods to find the eclipse period of our CV system.

## Methods Available:
- **Box Least Squares (BLS)**: Optimal for eclipse/transit detection
- **Lomb-Scargle (LS)**: General-purpose periodogram for regular variations
- **Auto-Correlation**: Cross-correlation based period detection

## Interactive Features:
- **Click-to-select**: Click on periodogram peaks to select periods
- **Dual-panel display**: View periodogram and folded light curve simultaneously
- **Method comparison**: Switch between BLS and LS methods
- **Statistical validation**: View period significance and false alarm probability

## Period Selection Strategy:
1. Start with **BLS method** for eclipse detection
2. Look for **significant peaks** (high power, low FAP)
3. **Visual inspection** of folded light curve
4. **Cross-validate** with Lomb-Scargle if needed
5. Consider **harmonics** and **aliases**

The interactive periodogram will display automatically, allowing you to explore different periods and immediately see the folded light curve results.

In [ ]:
# Template Parameters
TEMPLATE_PHASE_RANGE = 0.4
TEMPLATE_PHASE_RESOLUTION = 0.001
MIN_ECLIPSES_FOR_TEMPLATE = 5

print(f"Template construction:")
print(f"  Phase range: ±{TEMPLATE_PHASE_RANGE/2:.1f}")
print(f"  Resolution: {TEMPLATE_PHASE_RESOLUTION:.3f}")
print(f"  Min eclipses: {MIN_ECLIPSES_FOR_TEMPLATE}")

# Create template phase grid
template_phase_min = ECLIPSE_PHASE_CENTER - TEMPLATE_PHASE_RANGE/2
template_phase_max = ECLIPSE_PHASE_CENTER + TEMPLATE_PHASE_RANGE/2
template_phase_grid = np.arange(template_phase_min, template_phase_max + TEMPLATE_PHASE_RESOLUTION, 
                                TEMPLATE_PHASE_RESOLUTION)

print(f"✓ Template grid: {len(template_phase_grid)} points")

In [ ]:
# Interactive Period Search (O-C Maker v3 Style)
# ================================================

try:
    print("Creating interactive periodogram with click-to-select...")
    print("This may take a moment for large datasets...")
    
    # Create Lomb-Scargle periodogram
    print("  • Computing Lomb-Scargle periodogram...")
    periodogram = lc_clean.to_periodogram(method='lombscargle',
                                         minimum_period=period_min,
                                         maximum_period=period_max,
                                         oversample_factor=oversample_factor)
    
    # Extract data
    frequencies = periodogram.frequency.value
    power = periodogram.power.value
    
    # High-resolution interpolation for better peak detection
    n_samples = len(frequencies) * 2
    f_interp = np.linspace(frequencies.min(), frequencies.max(), n_samples)
    power_interp = np.interp(f_interp, frequencies, power)
    
    # Store basic results
    max_power = np.max(power)
    best_period = periodogram.period_at_max_power.value
    
    print(f"  • Periodogram computed with {len(frequencies)} frequency points")
    print(f"  • Maximum power: {max_power:.3f}")
    print(f"  • Best period (initial): {best_period:.6f} days")
    
    # Find peaks in the interpolated periodogram
    from scipy.signal import find_peaks
    peaks, peak_properties = find_peaks(power_interp, height=max_power/10)
    
    # Create DataFrame for easier handling
    peaks_df = pd.DataFrame({
        'frequencies': f_interp[peaks], 
        'power': power_interp[peaks], 
        'periods': 1/f_interp[peaks],
        'peak_index': peaks
    })
    
    # Sort by power (highest first)
    peaks_df = peaks_df.sort_values('power', ascending=False)
    
    print(f"  • Found {len(peaks_df)} significant peaks")
    print(f"  • Top 5 periods: {peaks_df['periods'].head().values}")
    
    # Create interactive FigureWidget (following O-C Maker v3 style)
    fi = go.FigureWidget([
        go.Scatter(
            x=peaks_df['frequencies'], 
            y=peaks_df['power'],
            mode='markers',
            name='Peaks',
            marker=dict(symbol='x', size=7, color='red'),
            hovertemplate='Frequency: %{x:.6f}<br>Power: %{y:.3f}<br>Period: %{customdata:.6f} days',
            customdata=peaks_df['periods']
        ),
        go.Scatter(
            x=frequencies, 
            y=power,
            mode='lines',
            name='Periodogram',
            line=dict(color='black')
        )
    ], layout=go.Layout(
        title=f'Interactive Periodogram - {TARGET_NAME} (Click on peaks to select)',
        xaxis_title='Frequency [1/d]',
        yaxis_title='Lomb-Scargle Power',
        hovermode='closest',
        height=600
    ))
    
    # Get the scatter trace for click events
    scatter = fi.data[0]
    
    # Initialize globals for click handling (following O-C Maker v3 pattern)
    global_periods = []
    global_periods_uncertainties = []
    
    # Color generator for multiple fits
    color_gen = (color for color in ['blue', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan'])
    
    # Define Gaussian function
    def gaussian(x, amplitude, mean, stddev):
        return amplitude * np.exp(-((x - mean) ** 2) / (2 * stddev ** 2))
    
    # Click event handler for interactive Gaussian fitting
    def update_point(trace, points, selector):
        if not points.point_inds:
            return
            
        # Get clicked point
        ind = points.point_inds[0]
        clicked_frequency = peaks_df.iloc[ind]['frequencies']
        clicked_period = peaks_df.iloc[ind]['periods']
        
        print(f"\n🖱️  CLICKED: Frequency = {clicked_frequency:.6f} cycles/day")
        print(f"   Period = {clicked_period:.6f} days")
        
        # Gaussian fitting (O-C Maker v3 style)
        try:
            # Define fitting window around clicked peak
            window_size = peak_width * 10  # Fitting window
            mask = (f_interp >= clicked_frequency - window_size) & (f_interp <= clicked_frequency + window_size)
            
            if np.sum(mask) < 10:
                print("   ❌ Insufficient data points for fitting")
                return
            
            f_fit = f_interp[mask]
            p_fit = power_interp[mask]
            
            # Initial guess for Gaussian parameters
            amplitude = np.max(p_fit)
            mean = clicked_frequency
            stddev = peak_width
            initial_guess = [amplitude, mean, stddev]
            
            print(f"   • Fitting Gaussian around peak...")
            print(f"   • Fitting window: ±{window_size:.6f} cycles/day")
            print(f"   • Data points in window: {len(f_fit)}")
            
            # Fit Gaussian
            from scipy.optimize import curve_fit
            popt, pcov = curve_fit(gaussian, f_fit, p_fit, p0=initial_guess)
            
            # Extract fitted parameters
            amp, cen, wid = popt
            
            # Calculate uncertainties
            perr = np.sqrt(np.diag(pcov))
            amp_err, cen_err, wid_err = perr
            
            # Convert to period and uncertainty
            fitted_period = 1 / cen
            period_uncertainty = (cen_err / cen**2)
            
            # Store results
            global_periods.append(fitted_period)
            global_periods_uncertainties.append(period_uncertainty)
            optimized_parameters_global.append(popt)
            
            # Generate fine grid for smooth curve
            f_fine = np.linspace(f_fit.min(), f_fit.max(), 200)
            fitted_gaussian_fine = gaussian(f_fine, amp, cen, wid)
            
            # Add fitted curve to plot
            color = next(color_gen)
            fi.add_trace(go.Scatter(
                x=f_fine,
                y=fitted_gaussian_fine,
                mode='lines',
                name=f'Fitted Gaussian {len(optimized_parameters_global)}',
                line=dict(color=color, width=2)
            ))
            
            print(f"  • Fitted frequency: {cen:.6f} cycles/day")
            print(f"  • Fitted period: {fitted_period:.6f} ± {period_uncertainty:.10f} days")
            print(f"  • Uncertainty in seconds: {period_uncertainty*24*60*60:.1f} s")
            
            # Create detailed plot
            fig_detail, ax = plt.subplots(figsize=(10, 6))
            ax.plot(f_interp, power_interp, label="Original Periodogram", color='black')
            ax.plot(f_fine, fitted_gaussian_fine, label='Fitted Gaussian', color='blue', linewidth=2)
            ax.plot(cen, amp, 'x', label='Peak', color='green', markersize=10)
            ax.plot(initial_guess[1], initial_guess[0], 'x', label='Initial Guess', color='red', markersize=8)
            ax.set_xlim(cen - 20*wid, cen + 20*wid)
            ax.set_title(f'Gaussian Fit for {TARGET_NAME}')
            ax.set_xlabel('Frequency [1/d]')
            ax.set_ylabel('Lomb-Scargle Power')
            ax.grid(True, linewidth=0.2)
            ax.legend(loc='upper right')
            
            # Add period information to plot
            ax.text(0.05, 0.95, f'Period =', transform=ax.transAxes, fontsize=12, 
                   verticalalignment='top', horizontalalignment='left')
            ax.text(0.05, 0.90, f'{fitted_period:.6f} ± {period_uncertainty:.10f} days', 
                   transform=ax.transAxes, fontsize=12, 
                   verticalalignment='top', horizontalalignment='left')
            
            plt.tight_layout()
            plt.show()
            
            # Update global selected period
            global selected_period, selected_method
            selected_period = fitted_period
            selected_method = f'Gaussian fit (click {len(optimized_parameters_global)})'
            
            print(f"✓ Period updated: {selected_period:.6f} days")
            
        except Exception as e:
            print(f"❌ Gaussian fit failed: {e}")
            # Fallback to simple peak selection
            selected_period = clicked_period
            selected_method = 'Peak selection (no Gaussian fit)'
            print(f"✓ Using peak period: {selected_period:.6f} days")
    
    # Attach click callback to scatter trace
    scatter.on_click(update_point)
    
    # Store periodogram data
    periodogram_results = {
        'frequencies': frequencies,
        'power': power,
        'f_interp': f_interp,
        'power_interp': power_interp,
        'peaks_df': peaks_df,
        'max_power': max_power,
        'peak_width': peak_width,
        'figure_widget': fi
    }
    
    print("✓ Interactive periodogram created successfully!")
    print("\nINSTRUCTIONS:")
    print("  1. The periodogram will appear below")
    print("  2. 🖱️  CLICK on any peak to select it and fit a Gaussian")
    print("  3. Multiple clicks will show different colored fits")
    print("  4. The last clicked peak will be used as the selected period")
    print("  5. Each click shows detailed Gaussian fit results")
    
    # Display the interactive plot
    try:
        display(fi)
    except:
        print("⚠️  Interactive plot not displaying properly in VS Code")
        print("   Creating static plot as fallback...")
        
        # Create static matplotlib plot as fallback
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Plot full periodogram
        ax.plot(frequencies, power, 'k-', alpha=0.7, label='Periodogram')
        
        # Plot peaks
        ax.scatter(peaks_df['frequencies'], peaks_df['power'], 
                  c='red', s=50, marker='x', label='Peaks', zorder=5)
        
        # Annotate top 5 peaks
        for i, (_, peak) in enumerate(peaks_df.head(5).iterrows()):
            ax.annotate(f'{peak["periods"]:.5f}d', 
                       (peak['frequencies'], peak['power']),
                       xytext=(5, 5), textcoords='offset points',
                       fontsize=8, ha='left')
        
        ax.set_xlabel('Frequency [1/d]')
        ax.set_ylabel('Lomb-Scargle Power')
        ax.set_title(f'Periodogram - {TARGET_NAME}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/periodogram_static.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("✓ Static periodogram displayed")
        print("  Top periods (click simulation):")
        for i, (_, peak) in enumerate(peaks_df.head(5).iterrows()):
            print(f"  {i+1}. {peak['periods']:.6f} days (power: {peak['power']:.3f})")
        
        # Auto-select best period for now
        best_period = peaks_df.iloc[0]['periods']
        selected_period = best_period
        selected_method = 'Auto-selected (fallback)'
        print(f"\n✓ Auto-selected best period: {selected_period:.6f} days")
    
    # Try to display interactive plot anyway
    fi
    
except Exception as e:
    print(f"❌ Error creating interactive periodogram: {e}")
    print("Falling back to basic periodogram analysis...")
    
    # Fallback: Basic BLS periodogram
    bls = lc_clean.to_periodogram(method='bls', 
                                  period_min=period_min,
                                  period_max=period_max,
                                  oversample_factor=oversample_factor)
    
    # Find best period
    best_period = bls.period_at_max_power.value
    best_power = bls.max_power.value
    
    print(f"  • Best BLS period: {best_period:.6f} days")
    print(f"  • BLS power: {best_power:.3f}")
    
    # Simple plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Periodogram
    ax1.plot(bls.period.value, bls.power.value, 'b-', alpha=0.7)
    ax1.axvline(best_period, color='r', linestyle='--', alpha=0.8)
    ax1.set_xlabel('Period (days)')
    ax1.set_ylabel('BLS Power')
    ax1.set_title(f'BLS Periodogram - {TARGET_NAME}')
    ax1.grid(True, alpha=0.3)
    
    # Folded light curve
    folded = lc_clean.fold(period=best_period)
    ax2.plot(folded.phase.value, folded.flux.value, 'bo', alpha=0.6, markersize=1)
    ax2.set_xlabel('Phase')
    ax2.set_ylabel('Normalized Flux')
    ax2.set_title(f'Folded at {best_period:.6f} days')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/basic_periodogram.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Store fallback results
    selected_period = best_period
    selected_method = 'BLS (fallback)'
    periodogram_results = {'bls_period': best_period, 'bls_power': best_power}

print(f"\n✓ Period search interface ready - CLICK ON PEAKS to select!")

In [ ]:
# Period Selection & Validation
# ==============================

# Check if period was selected from interactive plot
if 'selected_period' in globals() and selected_period is not None:
    period_selected = selected_period
    method_used = selected_method if selected_method else 'Interactive'
    
    print(f"✓ Period selected: {period_selected:.6f} days")
    print(f"  Method used: {method_used}")
    
else:
    # Manual period input option
    print("No period selected from interactive plot.")
    print("Please select a period from the periodogram above, or:")
    print("Run the cell below to input a period manually.")
    
    # For now, use the best period from periodogram if available
    if 'periodogram_results' in globals() and 'peaks_df' in periodogram_results:
        best_peak = periodogram_results['peaks_df'].iloc[0]
        period_selected = best_peak['periods']
        method_used = 'Lomb-Scargle (auto)'
        print(f"Using best Lomb-Scargle period: {period_selected:.6f} days")
    elif 'periodogram_results' in globals() and 'bls_period' in periodogram_results:
        period_selected = periodogram_results['bls_period']
        method_used = 'BLS (auto)'
        print(f"Using best BLS period: {period_selected:.6f} days")
    else:
        # Use default CV period as fallback
        period_selected = 0.1  # Default CV period
        method_used = 'Default CV period'
        print(f"Using default CV period: {period_selected:.6f} days")

# Store selected period
SELECTED_PERIOD = period_selected
PERIOD_METHOD = method_used

# Create validation plot
print(f"\nValidating selected period: {SELECTED_PERIOD:.6f} days")

# Create folded light curve for validation
folded = lc_clean.fold(period=SELECTED_PERIOD)

plt.figure(figsize=(10, 6))
plt.plot(folded.phase.value, folded.flux.value, 'bo', alpha=0.6, markersize=1)
plt.xlabel('Phase')
plt.ylabel('Normalized Flux')
plt.title(f'Folded Light Curve - Period: {SELECTED_PERIOD:.6f} days')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/folded_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Period validation complete")
print(f"  • Selected period: {SELECTED_PERIOD:.6f} days")
print(f"  • Period in hours: {SELECTED_PERIOD * 24:.2f} h")
print(f"  • Period in minutes: {SELECTED_PERIOD * 1440:.1f} min")
print(f"  • Method: {PERIOD_METHOD}")
print(f"  • Validation plot saved as 'folded_validation.png'")

In [ ]:
# Manual Period Override (Optional)
# ==================================

# Uncomment and modify the line below to manually set a period
# MANUAL_PERIOD = 0.123456  # Replace with your desired period in days

# Process manual period input
if 'MANUAL_PERIOD' in locals():
    SELECTED_PERIOD = MANUAL_PERIOD
    PERIOD_METHOD = 'Manual Input'
    
    # Validate manual period
    print(f"✓ Manual period override activated")
    print(f"  • Period: {SELECTED_PERIOD:.6f} days")
    print(f"  • Period in hours: {SELECTED_PERIOD * 24:.2f} h")
    print(f"  • Period in minutes: {SELECTED_PERIOD * 1440:.1f} min")
    
    # Create folded light curve for visual check
    folded = lc_clean.fold(period=SELECTED_PERIOD)
    
    plt.figure(figsize=(10, 6))
    plt.plot(folded.phase.value, folded.flux.value, 'bo', alpha=0.6, markersize=1)
    plt.xlabel('Phase')
    plt.ylabel('Normalized Flux')
    plt.title(f'Manual Period Check: {SELECTED_PERIOD:.6f} days')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/manual_period_check.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Manual period validation complete")
    print(f"  Visual check saved as 'manual_period_check.png'")
    
else:
    print("No manual period specified - using selected period from previous step")
    print("To use manual input, uncomment the MANUAL_PERIOD line above")

print(f"\nFinal period for analysis: {SELECTED_PERIOD:.6f} days")
print(f"Method: {PERIOD_METHOD}")
print(f"Ready to proceed with eclipse detection!")

# Step 5: Data Segmentation & Eclipse Detection 📊
---

Now that we have the orbital period, we'll segment the data and detect individual eclipses for timing analysis.

## Process Overview:
1. **Data Segmentation**: Divide the time series into orbital cycles
2. **Eclipse Detection**: Identify cycles containing eclipses
3. **Phase Folding**: Align eclipses to a common phase reference
4. **Quality Assessment**: Evaluate eclipse depth and shape consistency

## Key Parameters:
- **Eclipse Phase Window**: Defines the expected eclipse location (typically 0.4-0.6 in phase)
- **Detection Threshold**: Minimum flux drop required to classify as eclipse
- **Cycle Validation**: Ensures adequate data coverage per cycle

## Output:
- List of individual eclipse cycles with timestamps
- Eclipse detection statistics and quality metrics
- Phase-folded eclipse profiles for template construction

This step prepares the data for precise eclipse timing measurements.

In [ ]:
# Data Segmentation Parameters
# ============================

print("=" * 60)
print("STEP 5: Data Segmentation & Eclipse Detection")
print("=" * 60)

# Eclipse detection parameters
ECLIPSE_PHASE_WIDTH = 0.2   # Phase window around eclipse (±0.1 from center)
MIN_ECLIPSE_DEPTH = 0.01    # Minimum flux drop to classify as eclipse (1%)
MIN_CYCLE_COVERAGE = 0.7    # Minimum phase coverage required per cycle

print(f"Eclipse detection parameters:")
print(f"  • Eclipse phase width: ±{ECLIPSE_PHASE_WIDTH/2:.2f}")
print(f"  • Minimum eclipse depth: {MIN_ECLIPSE_DEPTH*100:.1f}%")
print(f"  • Minimum cycle coverage: {MIN_CYCLE_COVERAGE*100:.0f}%")

# Automatically detect eclipse phase from folded light curve
print(f"\nDetecting eclipse phase from folded light curve...")

# Create phase-folded light curve
folded = lk.LightCurve(time=time_q, flux=flux_q).fold(period=SELECTED_PERIOD)
phase_folded = folded.phase.value
flux_folded = folded.flux.value

# Sort by phase
sort_idx = np.argsort(phase_folded)
phase_folded = phase_folded[sort_idx]
flux_folded = flux_folded[sort_idx]

# Bin the folded light curve to find minimum
n_bins = 50
phase_bins = np.linspace(0, 1, n_bins)
binned_flux = np.zeros(n_bins - 1)

for i in range(n_bins - 1):
    mask = (phase_folded >= phase_bins[i]) & (phase_folded < phase_bins[i + 1])
    if np.sum(mask) > 0:
        binned_flux[i] = np.median(flux_folded[mask])
    else:
        binned_flux[i] = 1.0  # Default to normalized flux

# Find the phase of minimum flux (eclipse center)
min_idx = np.argmin(binned_flux)
ECLIPSE_PHASE_CENTER = (phase_bins[min_idx] + phase_bins[min_idx + 1]) / 2

print(f"  • Detected eclipse phase center: {ECLIPSE_PHASE_CENTER:.3f}")
print(f"  • Eclipse depth at detected phase: {(1 - binned_flux[min_idx])*100:.1f}%")

# Calculate eclipse phase window
eclipse_phase_min = ECLIPSE_PHASE_CENTER - ECLIPSE_PHASE_WIDTH/2
eclipse_phase_max = ECLIPSE_PHASE_CENTER + ECLIPSE_PHASE_WIDTH/2

# Handle phase wrapping around 1.0
if eclipse_phase_min < 0:
    eclipse_phase_min += 1.0
if eclipse_phase_max > 1:
    eclipse_phase_max -= 1.0

print(f"  • Eclipse phase window: {eclipse_phase_min:.3f} - {eclipse_phase_max:.3f}")

# Calculate cycle parameters
cycle_duration = SELECTED_PERIOD  # days
n_expected_cycles = int((time_q[-1] - time_q[0]) / cycle_duration)

print(f"\nData segmentation info:")
print(f"  • Selected period: {SELECTED_PERIOD:.6f} days")
print(f"  • Expected cycles: {n_expected_cycles}")
print(f"  • Time span: {time_q[-1] - time_q[0]:.2f} days")

# Initialize containers for cycle analysis
eclipse_cycles = []
cycle_info = []
eclipse_detected = []

In [ ]:
# Cycle Segmentation & Eclipse Detection
# ======================================

print("Segmenting data into orbital cycles...")

# Define reference epoch (first data point)
t0 = time_q[0]

# Segment data into cycles
cycle_count = 0
eclipse_count = 0

for cycle_num in range(n_expected_cycles):
    # Calculate cycle boundaries
    cycle_start = t0 + cycle_num * cycle_duration
    cycle_end = cycle_start + cycle_duration
    
    # Find data points in this cycle
    cycle_mask = (time_q >= cycle_start) & (time_q < cycle_end)
    
    if np.sum(cycle_mask) < 10:  # Skip cycles with too few points
        continue
    
    # Extract cycle data
    cycle_time = time_q[cycle_mask]
    cycle_flux = flux_q[cycle_mask]
    
    # Calculate phases for this cycle
    cycle_phase = ((cycle_time - cycle_start) / cycle_duration) % 1.0
    
    # Check phase coverage
    phase_coverage = (np.max(cycle_phase) - np.min(cycle_phase))
    if phase_coverage < MIN_CYCLE_COVERAGE:
        continue
    
    # Sort by phase
    phase_sort = np.argsort(cycle_phase)
    cycle_phase = cycle_phase[phase_sort]
    cycle_flux = cycle_flux[phase_sort]
    cycle_time = cycle_time[phase_sort]
    
    # Check for eclipse in expected phase range (handle phase wrapping)
    if eclipse_phase_min < eclipse_phase_max:
        # Normal case: eclipse window doesn't wrap around
        eclipse_mask = (cycle_phase >= eclipse_phase_min) & (cycle_phase <= eclipse_phase_max)
    else:
        # Phase wrapping case: eclipse window crosses phase=0/1 boundary
        eclipse_mask = (cycle_phase >= eclipse_phase_min) | (cycle_phase <= eclipse_phase_max)
    
    if np.sum(eclipse_mask) < 5:  # Need at least 5 points in eclipse window
        eclipse_detected.append(False)
    else:
        # Calculate eclipse depth
        eclipse_flux = cycle_flux[eclipse_mask]
        out_of_eclipse_flux = cycle_flux[~eclipse_mask]
        
        if len(out_of_eclipse_flux) > 0:
            baseline_flux = np.median(out_of_eclipse_flux)
            eclipse_depth = baseline_flux - np.min(eclipse_flux)
            eclipse_depth_fraction = eclipse_depth / baseline_flux
            
            # Check if eclipse is deep enough
            has_eclipse = eclipse_depth_fraction > MIN_ECLIPSE_DEPTH
            eclipse_detected.append(has_eclipse)
            
            if has_eclipse:
                eclipse_count += 1
        else:
            eclipse_detected.append(False)
    
    # Store cycle information
    cycle_info.append({
        'cycle_num': cycle_num,
        'cycle_start': cycle_start,
        'cycle_end': cycle_end,
        'n_points': len(cycle_time),
        'phase_coverage': phase_coverage,
        'has_eclipse': eclipse_detected[-1] if eclipse_detected else False,
        'eclipse_depth': eclipse_depth_fraction if 'eclipse_depth_fraction' in locals() else 0.0
    })
    
    # Store cycle data
    eclipse_cycles.append({
        'time': cycle_time,
        'flux': cycle_flux,
        'phase': cycle_phase,
        'cycle_start': cycle_start
    })
    
    cycle_count += 1

print(f"✓ Data segmentation complete!")
print(f"  • Total cycles processed: {cycle_count}")
print(f"  • Cycles with eclipses: {eclipse_count}")
print(f"  • Eclipse detection rate: {eclipse_count/cycle_count*100:.1f}%")

# Convert to arrays for easier handling
cycle_info = np.array(cycle_info)
eclipse_detected = np.array(eclipse_detected)

print(f"  • Mean eclipse depth: {np.mean([c['eclipse_depth'] for c in cycle_info if c['has_eclipse']])*100:.1f}%")
print(f"  • Mean phase coverage: {np.mean([c['phase_coverage'] for c in cycle_info])*100:.1f}%")

In [ ]:
# Visualize Eclipse Detection Results
# ===================================

print("Creating eclipse detection visualization...")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Eclipse detection timeline
eclipse_times = [c['cycle_start'] for c in cycle_info if c['has_eclipse']]
no_eclipse_times = [c['cycle_start'] for c in cycle_info if not c['has_eclipse']]

axes[0,0].plot(eclipse_times, np.ones(len(eclipse_times)), 'ro', markersize=8, alpha=0.7, label='Eclipse detected')
axes[0,0].plot(no_eclipse_times, np.zeros(len(no_eclipse_times)), 'bx', markersize=6, alpha=0.7, label='No eclipse')
axes[0,0].set_xlabel('Time (BJD_TDB)')
axes[0,0].set_ylabel('Eclipse Detection')
axes[0,0].set_title('Eclipse Detection Timeline')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_ylim(-0.5, 1.5)

# Plot 2: Eclipse depth distribution
eclipse_depths = [c['eclipse_depth'] for c in cycle_info if c['has_eclipse']]
if eclipse_depths:
    axes[0,1].hist(eclipse_depths, bins=20, alpha=0.7, color='green', edgecolor='black')
    axes[0,1].axvline(np.mean(eclipse_depths), color='red', linestyle='--', label=f'Mean: {np.mean(eclipse_depths)*100:.1f}%')
    axes[0,1].set_xlabel('Eclipse Depth (fraction)')
    axes[0,1].set_ylabel('Count')
    axes[0,1].set_title('Eclipse Depth Distribution')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)

# Plot 3: Phase coverage distribution
phase_coverages = [c['phase_coverage'] for c in cycle_info]
axes[1,0].hist(phase_coverages, bins=20, alpha=0.7, color='orange', edgecolor='black')
axes[1,0].axvline(np.mean(phase_coverages), color='red', linestyle='--', label=f'Mean: {np.mean(phase_coverages)*100:.1f}%')
axes[1,0].axvline(MIN_CYCLE_COVERAGE, color='black', linestyle=':', label=f'Min required: {MIN_CYCLE_COVERAGE*100:.0f}%')
axes[1,0].set_xlabel('Phase Coverage')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Cycle Phase Coverage')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Sample eclipse profiles
if eclipse_count > 0:
    eclipse_indices = [i for i, c in enumerate(cycle_info) if c['has_eclipse']]
    n_samples = min(10, len(eclipse_indices))  # Show up to 10 examples
    
    for i in range(n_samples):
        idx = eclipse_indices[i]
        cycle_data = eclipse_cycles[idx]
        axes[1,1].plot(cycle_data['phase'], cycle_data['flux'], '-o', alpha=0.6, markersize=2)
    
    axes[1,1].axvspan(eclipse_phase_min, eclipse_phase_max, alpha=0.2, color='red', label='Eclipse window')
    axes[1,1].set_xlabel('Phase')
    axes[1,1].set_ylabel('Normalized Flux')
    axes[1,1].set_title(f'Sample Eclipse Profiles (n={n_samples})')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    axes[1,1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eclipse_detection_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Eclipse detection visualization saved as 'eclipse_detection_results.png'")

# Summary statistics
print(f"\nEclipse Detection Summary:")
print(f"  • Total cycles analyzed: {len(cycle_info)}")
print(f"  • Cycles with eclipses: {eclipse_count}")
print(f"  • Detection success rate: {eclipse_count/len(cycle_info)*100:.1f}%")
print(f"  • Mean eclipse depth: {np.mean(eclipse_depths)*100:.1f}% ± {np.std(eclipse_depths)*100:.1f}%" if eclipse_depths else "No eclipses detected")
print(f"  • Ready for template construction")

# Step 6: Template Construction 🔨
---

Create a high-quality eclipse template by combining multiple eclipse profiles. This template will be used for precise timing measurements via cross-correlation.

## Template Construction Strategy:
1. **Eclipse Selection**: Choose high-quality eclipses with good coverage and depth
2. **Phase Alignment**: Align all eclipses to a common phase grid
3. **Outlier Rejection**: Remove cycles with unusual eclipse shapes or noise
4. **Weighted Averaging**: Combine eclipses with appropriate weights (e.g., by S/N)
5. **Smoothing**: Apply optional smoothing to reduce noise while preserving eclipse features

## Quality Criteria:
- **Minimum Eclipse Depth**: Ensures detectable eclipse features
- **Phase Coverage**: Requires adequate sampling around eclipse
- **Shape Consistency**: Filters out unusual or corrupted eclipses
- **Signal-to-Noise**: Prioritizes high-quality data

## Template Features:
- **High-resolution phase grid** for precise timing
- **Uncertainty estimates** for each phase bin
- **Quality metrics** and diagnostic information
- **Visual validation** of template shape and consistency

The resulting template will serve as the reference for measuring eclipse mid-times in each cycle.

In [ ]:
# This cell has been removed - parameters moved to previous cell

In [ ]:
# Build Eclipse Template
print("Building eclipse template...")

# Collect eclipse data
eclipse_data_list = []
for i, cycle_data in enumerate(eclipse_cycles):
    if not cycle_info[i]['quality_good']:
        continue
    
    phase = cycle_data['phase']
    flux = cycle_data['flux']
    
    # Focus on eclipse region
    eclipse_mask = (phase >= template_phase_min) & (phase <= template_phase_max)
    if np.sum(eclipse_mask) < 10:
        continue
    
    eclipse_phase = phase[eclipse_mask]
    eclipse_flux = flux[eclipse_mask]
    eclipse_depth = cycle_info[i]['eclipse_depth']
    
    eclipse_data_list.append({
        'phase': eclipse_phase,
        'flux': eclipse_flux,
        'depth': eclipse_depth,
        'weight': eclipse_depth / cycle_info[i]['flux_scatter']
    })

# Build template by weighted binning
template_flux = np.ones_like(template_phase_grid)
template_uncertainty = np.ones_like(template_phase_grid) * 0.1

for i, phase_center in enumerate(template_phase_grid):
    values = []
    weights = []
    
    for eclipse_data in eclipse_data_list:
        mask = np.abs(eclipse_data['phase'] - phase_center) < TEMPLATE_PHASE_RESOLUTION
        if np.sum(mask) > 0:
            values.extend(eclipse_data['flux'][mask])
            weights.extend([eclipse_data['weight']] * np.sum(mask))
    
    if values:
        template_flux[i] = np.average(values, weights=weights)
        template_uncertainty[i] = np.std(values) / np.sqrt(len(values))

# Normalize template
template_flux = template_flux / np.median(template_flux)
eclipse_depth = 1 - template_flux.min()

eclipse_template = {
    'phase': template_phase_grid,
    'flux': template_flux,
    'uncertainty': template_uncertainty,
    'depth': eclipse_depth,
    'n_eclipses': len(eclipse_data_list)
}

print(f"✓ Template built from {len(eclipse_data_list)} eclipses")
print(f"  Eclipse depth: {eclipse_depth:.3f}")

In [ ]:
# Template Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Template
axes[0].errorbar(eclipse_template['phase'], eclipse_template['flux'], 
                 yerr=eclipse_template['uncertainty'], 
                 fmt='o-', color='red', markersize=2, linewidth=2, alpha=0.8)
axes[0].set_xlabel('Phase')
axes[0].set_ylabel('Normalized Flux')
axes[0].set_title(f'Eclipse Template (n={eclipse_template["n_eclipses"]})')
axes[0].grid(True, alpha=0.3)

# Plot 2: Individual eclipses vs template
for i, eclipse_data in enumerate(eclipse_data_list[:5]):
    axes[1].plot(eclipse_data['phase'], eclipse_data['flux'], 
                 'o-', alpha=0.5, markersize=2, linewidth=1)
axes[1].plot(eclipse_template['phase'], eclipse_template['flux'], 
             'k-', linewidth=3, alpha=0.8, label='Template')
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Normalized Flux')
axes[1].set_title('Individual Eclipses vs Template')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✓ Template ready for timing measurements")

# Step 7: Eclipse Mid-time Measurements ⏱️
---

Measure precise eclipse mid-times using template cross-correlation. This is the core timing analysis that will feed into the O-C diagram.

## Cross-Correlation Method:
1. **Template Matching**: Compare each eclipse cycle with the constructed template
2. **Phase Shift Optimization**: Find the phase shift that maximizes correlation
3. **Uncertainty Estimation**: Calculate timing uncertainties from fit quality
4. **Quality Assessment**: Evaluate correlation strength and consistency

## Key Parameters:
- **Shift Range**: Maximum phase shift to search (typically ±0.02 in phase)
- **Shift Resolution**: Step size for phase shift search
- **Quality Thresholds**: Minimum correlation coefficient and S/N requirements
- **Outlier Detection**: Identify and flag problematic measurements

## Timing Precision:
- **Template Resolution**: Limited by template phase sampling
- **Data Quality**: Depends on eclipse depth and photometric precision
- **Systematic Errors**: Addressed through quality cuts and uncertainty estimates

## Output:
- **Eclipse Mid-times**: Precise timing measurements for each cycle
- **Timing Uncertainties**: Statistical error estimates
- **Quality Flags**: Indicators of measurement reliability
- **Correlation Statistics**: Diagnostics for template matching quality

These measurements will form the basis for the O-C analysis in the next steps.

In [ ]:
# Eclipse Timing Parameters
MAX_PHASE_SHIFT = 0.02
PHASE_SHIFT_RESOLUTION = 0.001
MIN_CORRELATION_COEFF = 0.5

phase_shifts = np.arange(-MAX_PHASE_SHIFT, MAX_PHASE_SHIFT + PHASE_SHIFT_RESOLUTION, PHASE_SHIFT_RESOLUTION)

print(f"Timing parameters:")
print(f"  Phase shift range: ±{MAX_PHASE_SHIFT:.3f}")
print(f"  Resolution: {PHASE_SHIFT_RESOLUTION:.3f}")
print(f"  Min correlation: {MIN_CORRELATION_COEFF:.2f}")
print(f"  Grid points: {len(phase_shifts)}")

In [ ]:
# Measure Eclipse Mid-times
print("Measuring eclipse mid-times...")

eclipse_midtimes = []
timing_uncertainties = []
correlation_coeffs = []
cycle_numbers = []

for i, eclipse_data in enumerate(eclipse_data_list):
    cycle_num = eclipse_data.get('cycle_num', i)
    
    try:
        # Find corresponding cycle
        cycle_start_time = eclipse_cycles[cycle_num]['cycle_start'] if cycle_num < len(eclipse_cycles) else eclipse_cycles[i]['cycle_start']
        
        # Use cross-correlation to find best timing
        midtime, uncertainty, correlation = measure_midpoints(
            eclipse_data['phase'], eclipse_data['flux'], 
            eclipse_template['phase'], eclipse_template['flux'],
            phase_shifts=phase_shifts,
            reference_time=cycle_start_time,
            period=SELECTED_PERIOD
        )
        
        if correlation >= MIN_CORRELATION_COEFF:
            eclipse_midtimes.append(midtime)
            timing_uncertainties.append(uncertainty)
            correlation_coeffs.append(correlation)
            cycle_numbers.append(cycle_num)
            
    except Exception as e:
        continue

eclipse_midtimes = np.array(eclipse_midtimes)
timing_uncertainties = np.array(timing_uncertainties)
correlation_coeffs = np.array(correlation_coeffs)
cycle_numbers = np.array(cycle_numbers)

print(f"✓ Successfully measured {len(eclipse_midtimes)} eclipse times")
print(f"  Mean correlation: {np.mean(correlation_coeffs):.3f}")
print(f"  Mean uncertainty: {np.mean(timing_uncertainties)*24*60:.1f} minutes")

In [ ]:
# O-C Analysis
print("Computing O-C diagram...")

# Calculate expected eclipse times
t0 = eclipse_midtimes[0]  # Reference time
expected_times = t0 + cycle_numbers * SELECTED_PERIOD

# Calculate O-C residuals
oc_residuals = (eclipse_midtimes - expected_times) * 24 * 60  # Convert to minutes

# Remove outliers
outlier_mask = np.abs(oc_residuals - np.median(oc_residuals)) < TIMING_OUTLIER_SIGMA * np.std(oc_residuals)
oc_clean = oc_residuals[outlier_mask]
times_clean = eclipse_midtimes[outlier_mask]
cycles_clean = cycle_numbers[outlier_mask]

print(f"✓ O-C analysis complete")
print(f"  Total measurements: {len(eclipse_midtimes)}")
print(f"  After outlier removal: {len(oc_clean)}")
print(f"  RMS scatter: {np.std(oc_clean):.1f} minutes")
print(f"  Time span: {times_clean.max() - times_clean.min():.1f} days")

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# O-C diagram
axes[0].errorbar(times_clean, oc_clean, yerr=np.array(timing_uncertainties)[outlier_mask]*24*60,
                fmt='o', markersize=3, alpha=0.7, color='blue')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time (BJD_TDB)')
axes[0].set_ylabel('O-C (minutes)')
axes[0].set_title(f'{TARGET_NAME} - O-C Diagram')
axes[0].grid(True, alpha=0.3)

# Timing residuals vs cycle
axes[1].plot(cycles_clean, oc_clean, 'o-', markersize=3, alpha=0.7, color='green')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Cycle Number')
axes[1].set_ylabel('O-C (minutes)')
axes[1].set_title('O-C vs Cycle Number')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save results
np.savetxt(f'{OUTPUT_DIR}/eclipse_times.csv', 
           np.column_stack([times_clean, cycles_clean, oc_clean, 
                           np.array(timing_uncertainties)[outlier_mask]*24*60]),
           header='BJD_TDB,Cycle,OC_min,Uncertainty_min',
           delimiter=',', fmt='%.6f')

print(f"✓ Results saved to {OUTPUT_DIR}/eclipse_times.csv")
print(f"✓ Analysis complete!")